# ClickTV Google Colab — Final Easy 5 Mode

শুধু ৫টি mode:

- `channels` — শুধু TV Channels
- `today` — শুধু Today Match
- `upcoming` — শুধু Upcoming Match
- `movies` — শুধু Movies
- `all` — উপরের সব একসঙ্গে

ব্যবহার:

1. Cell 1 একবার চালান।
2. Cell 2-তে একটি mode বেছে Play চাপুন।
3. Scan শেষ হলে GitHub push নিজে হবে।
4. GitHub push সফল হওয়ার পরেই Telegram message যাবে।
5. Push ব্যর্থ হলে একই Cell 2 আবার চালালে pending result আগে auto-push হবে।

আলাদা recovery cell, push checkbox, events/discovery mode বা extra code নেই।


In [1]:
#@title 1) Connect GitHub safely
import getpass
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

REPO_OWNER = "DigeeGlamour"
REPO_NAME = "click-tv"
BRANCH = "main"
REPO_DIR = Path("/content") / REPO_NAME
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

try:
    from google.colab import userdata
except Exception:
    userdata = None


def get_secret(name: str, required: bool = False) -> str:
    value = ""
    if userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""

    if required and not value:
        value = getpass.getpass(f"Enter {name}: ").strip()

    return value.strip()


GITHUB_TOKEN = get_secret("GITHUB_TOKEN", required=True)
PRIVATE_MOVIE_SOURCE_TOKEN = get_secret("PRIVATE_MOVIE_SOURCE_TOKEN") or GITHUB_TOKEN
TMDB_API_KEY = get_secret("TMDB_API_KEY")
TMDB_API_TOKEN = get_secret("TMDB_API_TOKEN")
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = get_secret("TELEGRAM_CHAT_ID")
RUNTIME_SECRETS = [
    value for value in (
        GITHUB_TOKEN, PRIVATE_MOVIE_SOURCE_TOKEN, TMDB_API_KEY,
        TMDB_API_TOKEN, TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID,
    ) if len(value) >= 8
]


def redact_runtime_secrets(value):
    clean = str(value)
    for secret in sorted(set(RUNTIME_SECRETS), key=len, reverse=True):
        clean = clean.replace(secret, "[REDACTED]")
    return clean

if not GITHUB_TOKEN:
    raise RuntimeError(
        "Colab Secrets-এ GITHUB_TOKEN যোগ করে Notebook access ON করুন।"
    )

if any(character.isspace() for character in GITHUB_TOKEN):
    raise RuntimeError(
        "GITHUB_TOKEN-এর Value-তে space/newline আছে।"
    )

if GITHUB_TOKEN.count("github_pat_") > 1:
    raise RuntimeError(
        "Token value ভুল: github_pat_ prefix একাধিকবার আছে।"
    )

if not (
    GITHUB_TOKEN.startswith("github_pat_")
    or GITHUB_TOKEN.startswith("ghp_")
):
    raise RuntimeError(
        "GITHUB_TOKEN সঠিক GitHub PAT format নয়।"
    )


def validate_github_token(token: str) -> str:
    request = urllib.request.Request(
        "https://api.github.com/user",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "User-Agent": "ClickTV-Colab-Scanner",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError(
                "GitHub token invalid বা expired।"
            ) from error
        raise RuntimeError(
            f"GitHub token validation failed: HTTP {error.code}"
        ) from error

    login = str(payload.get("login") or "").strip()
    if not login:
        raise RuntimeError("GitHub account verify করা যায়নি।")
    return login


authenticated_login = validate_github_token(GITHUB_TOKEN)
print(f"✅ GitHub token verified: {authenticated_login}")

ASKPASS_PATH = Path(tempfile.gettempdir()) / "clicktv_git_askpass.sh"
ASKPASS_PATH.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;
  *) printf '%s\n' '' ;;
esac
""",
    encoding="utf-8",
)
ASKPASS_PATH.chmod(
    ASKPASS_PATH.stat().st_mode
    | stat.S_IXUSR
    | stat.S_IXGRP
    | stat.S_IXOTH
)

GIT_ENV = os.environ.copy()
GIT_ENV["GIT_ASKPASS"] = str(ASKPASS_PATH)
GIT_ENV["GIT_TERMINAL_PROMPT"] = "0"
GIT_ENV["GITHUB_TOKEN"] = GITHUB_TOKEN


def run_git(args, cwd=None, check=True):
    result = subprocess.run(
        ["git"] + list(args),
        cwd=str(cwd) if cwd else None,
        env=GIT_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.stdout.strip():
        print(result.stdout.rstrip())

    if result.returncode != 0:
        safe_error = redact_runtime_secrets(result.stderr)
        if safe_error.strip():
            print(safe_error.rstrip())

        if check:
            raise RuntimeError(
                f"Git command failed: git {' '.join(args)}"
            )

    return result


# Always begin with a clean clone.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("Cloning latest repository...")
run_git(
    [
        "clone",
        "--branch",
        BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ]
)

run_git(["config", "user.name", "Live Signal Colab"], cwd=REPO_DIR)
run_git(
    ["config", "user.email", "colab-scanner@users.noreply.github.com"],
    cwd=REPO_DIR,
)

commit_id = run_git(
    ["rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
).stdout.strip()

print("=" * 68)
print("✅ READY")
print(f"Repository : {REPO_OWNER}/{REPO_NAME}")
print(f"Branch     : {BRANCH}")
print(f"Commit     : {commit_id}")
print("=" * 68)


✅ GitHub token verified: DigeeGlamour
Cloning latest repository...
bef0fcc
✅ READY
Repository : DigeeGlamour/click-tv
Branch     : main
Commit     : bef0fcc


In [2]:
#@title 2) Select one mode and run automatically
SCAN_MODE = "movies" #@param ["channels", "today", "upcoming", "movies", "all"]

import json
import os
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

SUPPORTED_MODES = {"channels", "today", "upcoming", "movies", "all"}
if SCAN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported mode: {SCAN_MODE}")

PROGRESS_EVERY = 25
HEARTBEAT_SECONDS = 20
log_path = Path("/content/clicktv-scanner.log")


def clean_temporary_files():
    subprocess.run(
        ["git", "restore", "--staged", "--worktree", "--", "."],
        cwd=REPO_DIR,
        check=False,
    )

    checkpoint = REPO_DIR / "working" / "pipeline-checkpoint.json"
    if checkpoint.exists():
        try:
            checkpoint.unlink()
        except OSError:
            pass

    progress = REPO_DIR / "working" / "scan-progress.json"
    if progress.exists():
        try:
            progress.unlink()
        except OSError:
            pass

    checkpoints_dir = REPO_DIR / "working" / "checkpoints"
    if checkpoints_dir.exists():
        shutil.rmtree(checkpoints_dir, ignore_errors=True)

    source_cache_dir = REPO_DIR / "working" / "source-cache"
    if source_cache_dir.exists():
        shutil.rmtree(source_cache_dir, ignore_errors=True)

    for cache_dir in REPO_DIR.rglob("__pycache__"):
        if cache_dir.is_dir():
            shutil.rmtree(cache_dir, ignore_errors=True)

    for pyc_file in REPO_DIR.rglob("*.pyc"):
        try:
            pyc_file.unlink()
        except OSError:
            pass


def push_with_retry(max_attempts=3):
    run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_git(["rebase", f"origin/{BRANCH}"], cwd=REPO_DIR)

    for attempt in range(1, max_attempts + 1):
        result = run_git(
            ["push", "origin", f"HEAD:{BRANCH}"],
            cwd=REPO_DIR,
            check=False,
        )
        if result.returncode == 0:
            return

        if attempt == max_attempts:
            raise RuntimeError(
                f"GitHub push {max_attempts} বার চেষ্টা করেও সফল হয়নি। "
                "একই Cell 2 আবার চালালে pending result আগে auto-push হবে।"
            )

        print(f"Push retry {attempt}/{max_attempts} failed; retrying...")
        time.sleep(5)
        run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR)
        run_git(["rebase", f"origin/{BRANCH}"], cwd=REPO_DIR)


def send_post_push_notification(mode, commit):
    notify_environment = os.environ.copy()
    if TELEGRAM_BOT_TOKEN:
        notify_environment["TELEGRAM_BOT_TOKEN"] = TELEGRAM_BOT_TOKEN
    if TELEGRAM_CHAT_ID:
        notify_environment["TELEGRAM_CHAT_ID"] = TELEGRAM_CHAT_ID

    result = subprocess.run(
        [
            sys.executable,
            "-u",
            "scanner/telegram_notify.py",
            mode,
            commit,
            BRANCH,
        ],
        cwd=REPO_DIR,
        env=notify_environment,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.stdout.strip():
        print(result.stdout.rstrip())


# ------------------------------------------------------------------
# If an earlier scan finished but push failed, recover it automatically.
# No third cell and no extra recovery code are needed.
# ------------------------------------------------------------------
run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR)

ahead_result = subprocess.run(
    ["git", "rev-list", "--count", f"origin/{BRANCH}..HEAD"],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
ahead_count = int((ahead_result.stdout or "0").strip() or "0")

if ahead_count > 0:
    print(
        f"Found {ahead_count} pending scan commit(s). "
        "আগে সেগুলো auto-push করা হচ্ছে..."
    )
    clean_temporary_files()

    remaining = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    ).stdout.strip()

    if remaining:
        print(remaining)
        raise RuntimeError(
            "Pending result push করার আগে repository clean হয়নি।"
        )

    push_with_retry()
    pending_commit = run_git(
        ["rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
    ).stdout.strip()

    summary_path = REPO_DIR / "reports" / "scan-summary.json"
    pending_mode = SCAN_MODE
    try:
        pending_summary = json.loads(
            summary_path.read_text(encoding="utf-8")
        )
        pending_mode = str(
            pending_summary.get("mode") or SCAN_MODE
        ).strip().lower()
    except Exception:
        pass

    if pending_mode not in SUPPORTED_MODES:
        pending_mode = SCAN_MODE

    send_post_push_notification(pending_mode, pending_commit)

    print("=" * 68)
    print("✅ PREVIOUS SCAN RESULT AUTO-PUSHED")
    print(f"Commit : {pending_commit}")
    print("নতুন scan চালানো হয়নি। Cell 2 আবার চাপলে selected mode scan হবে।")
    print("=" * 68)
else:
    # Always start this scan from latest main.
    run_git(["checkout", BRANCH], cwd=REPO_DIR)
    run_git(["reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

    required_paths = [
        "scan.py",
        "config/settings.json",
        "config/sources.json",
        "scanner/content_router.py",
        "scanner/fast_pipeline.py",
        "scanner/source_loader.py",
        "scanner/normalizer.py",
        "scanner/planner.py",
        "scanner/verifier.py",
        "scanner/bd_verifier.py",
        "scanner/channels.py",
        "scanner/movies.py",
        "scanner/events.py",
        "scanner/output.py",
        "scanner/telegram_notify.py",
    ]

    missing = [
        path for path in required_paths
        if not (REPO_DIR / path).exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Required files missing:\n- " + "\n- ".join(missing)
        )

    print("[PREFLIGHT] Checking scanner code...")
    compile_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "py_compile",
            "scan.py",
            *[
                str(path.relative_to(REPO_DIR))
                for path in (REPO_DIR / "scanner").glob("*.py")
            ],
            *[
                str(path.relative_to(REPO_DIR))
                for path in (REPO_DIR / "scanner" / "parsers").glob("*.py")
            ],
        ],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if compile_result.stdout.strip():
        print(compile_result.stdout.rstrip())
    if compile_result.returncode != 0:
        raise RuntimeError("Python syntax validation failed.")

    settings_path = REPO_DIR / "config" / "settings.json"
    original_settings = settings_path.read_text(encoding="utf-8")
    settings = json.loads(original_settings)
    verification = settings.setdefault("verification", {})
    verification["progress_interval"] = PROGRESS_EVERY
    settings_path.write_text(
        json.dumps(settings, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    scan_environment = os.environ.copy()
    scan_environment["PYTHONUNBUFFERED"] = "1"
    scan_environment["PYTHONFAULTHANDLER"] = "1"
    scan_environment["PRIVATE_MOVIE_SOURCE_TOKEN"] = PRIVATE_MOVIE_SOURCE_TOKEN
    if TMDB_API_KEY:
        scan_environment["TMDB_API_KEY"] = TMDB_API_KEY
    if TMDB_API_TOKEN:
        scan_environment["TMDB_API_TOKEN"] = TMDB_API_TOKEN
    # Telegram secrets are intentionally NOT passed to the scanner.
    # Notification is sent only after GitHub push succeeds.

    print("\n" + "=" * 72)
    print("CLICK TV SCANNER")
    print(f"Mode        : {SCAN_MODE}")
    print("Auto Push   : ON")
    print("Telegram    : After successful GitHub push only")
    print("=" * 72)

    process = subprocess.Popen(
        [sys.executable, "-u", "scan.py", SCAN_MODE],
        cwd=REPO_DIR,
        env=scan_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    stop_monitor = threading.Event()
    started = time.monotonic()
    last_activity = {"text": "Starting scanner...", "changed_at": time.monotonic()}
    progress_path = REPO_DIR / "working" / "scan-progress.json"
    checkpoint_path = REPO_DIR / "working" / "pipeline-checkpoint.json"
    last_progress_signature = {"value": ""}

    def monitor():
        while not stop_monitor.wait(HEARTBEAT_SECONDS):
            elapsed = int(time.monotonic() - started)
            minutes, seconds = divmod(elapsed, 60)
            progress_text = ""
            try:
                progress = json.loads(progress_path.read_text(encoding="utf-8"))
                signature = json.dumps(progress, sort_keys=True, ensure_ascii=False)
                stage = str(progress.get("stage") or "unknown")
                current = str(progress.get("current_pipeline") or progress.get("mode") or SCAN_MODE)
                pipeline_index = progress.get("pipeline_index")
                pipeline_total = progress.get("pipeline_total")
                order_text = f" {pipeline_index}/{pipeline_total}" if pipeline_index and pipeline_total else ""
                progress_text = f"stage={stage} | pipeline={current}{order_text}"
                if checkpoint_path.exists():
                    checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
                    checkpoint_signature = json.dumps(checkpoint, sort_keys=True, ensure_ascii=False)
                    signature += checkpoint_signature
                    progress_text += (f" | global={checkpoint.get('global_completed', 0)} "
                                      f"queued={checkpoint.get('pending_global', 0)} "
                                      f"proxy={checkpoint.get('bd_completed', 0)}/{checkpoint.get('bd_selected', 0)}")
                if signature != last_progress_signature["value"]:
                    last_progress_signature["value"] = signature
                    last_activity["changed_at"] = time.monotonic()
            except Exception:
                progress_text = last_activity['text'][:180]
            idle_seconds = int(time.monotonic() - last_activity["changed_at"])
            warning = f" | WARNING: no structured progress for {idle_seconds}s" if idle_seconds >= 180 else ""
            print(
                f"\n[MONITOR {minutes:02d}:{seconds:02d}] "
                f"Scanner running | {progress_text}{warning}"
            )
            sys.stdout.flush()

    monitor_thread = threading.Thread(target=monitor, daemon=True)
    monitor_thread.start()

    try:
        with log_path.open("w", encoding="utf-8") as log_handle:
            assert process.stdout is not None
            for raw_line in process.stdout:
                line = raw_line.rstrip("\n")
                timestamp = datetime.now().strftime("%H:%M:%S")
                rendered = redact_runtime_secrets(f"[{timestamp}] {line}")
                print(rendered, flush=True)
                log_handle.write(rendered + "\n")
                log_handle.flush()
                if line.strip():
                    last_activity["text"] = line.strip()
                    last_activity["changed_at"] = time.monotonic()

        return_code = process.wait()
    finally:
        stop_monitor.set()
        monitor_thread.join(timeout=3)
        settings_path.write_text(original_settings, encoding="utf-8")

    if return_code != 0:
        print("\n❌ Scanner failed. GitHub push and Telegram were not run.")
        raise RuntimeError(
            f"Scanner failed with exit code {return_code}"
        )

    print("\n[PUSH] Committing generated data...")
    subprocess.run(
        ["git", "add", "-A", "--", "data", "reports", "state"],
        cwd=REPO_DIR,
        check=True,
    )

    has_changes = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
    ).returncode != 0

    if has_changes:
        timestamp = datetime.now(timezone.utc).strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )
        subprocess.run(
            [
                "git",
                "commit",
                "-m",
                f"Colab update: {SCAN_MODE} [{timestamp}]",
            ],
            cwd=REPO_DIR,
            check=True,
        )
    else:
        print("No output changes found.")

    clean_temporary_files()

    remaining = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    ).stdout.strip()

    if remaining:
        print("Unexpected remaining changes:")
        print(remaining)
        raise RuntimeError(
            "Repository clean হয়নি। GitHub push ও Telegram বন্ধ রাখা হয়েছে।"
        )

    print("[PUSH] Uploading to GitHub automatically...")
    push_with_retry()

    pushed_commit = run_git(
        ["rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
    ).stdout.strip()

    print("[TELEGRAM] GitHub push successful; sending message now...")
    send_post_push_notification(SCAN_MODE, pushed_commit)

    print("\n" + "=" * 72)
    print("✅ SCAN + GITHUB PUSH COMPLETED")
    print(f"Mode   : {SCAN_MODE}")
    print(f"Commit : {pushed_commit}")
    print("Telegram message was attempted only after successful push.")
    print("=" * 72)


Your branch is up to date with 'origin/main'.
HEAD is now at bef0fcc Update settings.json
[PREFLIGHT] Checking scanner code...

CLICK TV SCANNER
Mode        : movies
Auto Push   : ON
Telegram    : After successful GitHub push only
[16:08:37] ==================================================
[16:08:37] 🚀 LIVE SIGNAL SCANNER - STARTING MODE: MOVIES
[16:08:37] ==================================================
[16:08:37] 
[16:08:37] [Step 1a/5] Fetching sources and detecting formats...

[MONITOR 00:20] Scanner running | [Step 1a/5] Fetching sources and detecting formats...

[MONITOR 00:40] Scanner running | [Step 1a/5] Fetching sources and detecting formats...
[16:09:33] 
[16:09:33] [Step 1b/5] Normalization completed...
[16:09:33]    Candidates: 71885 raw -> 71868 normalized -> 5000 candidate pool
[16:09:33]    Adaptive first wave: 5000 candidates; later candidates run only when needed
[16:09:33]    Planner reductions: exact duplicates=29872, unknown TV=0, per-item cap=201, global cap=3